In [1]:
 import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Load and preprocess the CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

x_train = x_train.astype('float32') / 255.0
x_val = x_val.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train = to_categorical(y_train, 10)
y_val = to_categorical(y_val, 10)
y_test = to_categorical(y_test, 10)

# Define a function to create a CNN model
def create_cnn_model(learning_rate=0.001, filter_size=32, num_layers=3, dropout_rate=0.5, optimizer='adam'):
    model = models.Sequential()
    model.add(layers.Conv2D(filter_size, (3, 3), activation='relu', input_shape=(32, 32, 3)))
    model.add(layers.MaxPooling2D((2, 2)))

    for _ in range(num_layers - 1):
        model.add(layers.Conv2D(filter_size * 2, (3, 3), activation='relu'))
        model.add(layers.MaxPooling2D((2, 2)))
        filter_size *= 2

    model.add(layers.Flatten())
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(10, activation='softmax'))

    if optimizer == 'adam':
        opt = optimizers.Adam(learning_rate=learning_rate)
    elif optimizer == 'sgd':
        opt = optimizers.SGD(learning_rate=learning_rate)
    else:
        raise ValueError("Unsupported optimizer type")

    model.compile(optimizer=opt,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Create and train the model
model = create_cnn_model(learning_rate=0.001, filter_size=32, num_layers=3, dropout_rate=0.5, optimizer='adam')
history = model.fit(x_train, y_train, epochs=10, batch_size=64, validation_data=(x_val, y_val))

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f'Test accuracy: {test_acc}')


/opt/anaconda3/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:99: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(


Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 17s 26ms/step - accuracy: 0.2408 - loss: 2.0140 - val_accuracy: 0.4494 - val_loss: 1.5150
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.4521 - loss: 1.5140 - val_accuracy: 0.5337 - val_loss: 1.2882
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 24s 38ms/step - accuracy: 0.5107 - loss: 1.3604 - val_accuracy: 0.5917 - val_loss: 1.1542
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.5540 - loss: 1.2541 - val_accuracy: 0.6190 - val_loss: 1.0713
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.5910 - loss: 1.1563 - val_accuracy: 0.6494 - val_loss: 1.0015
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.6179 - loss: 1.0833 - val_accuracy: 0.6453 - val_loss: 1.0035
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.6435 - loss: 1.0188 - val_accuracy: 0.6584 - val_loss: 0.9684
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 22s 34ms/step - accuracy: 0.6600 - loss: 0.9718 - 

In [2]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')

# Make predictions on the test set
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

# Print classification report
print("Classification Report:")
print(classification_report(y_true, y_pred_classes))

# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred_classes))


Test accuracy: 0.6888
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step
Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.78      0.74      1000
           1       0.81      0.86      0.83      1000
           2       0.51      0.63      0.57      1000
           3       0.50      0.43      0.46      1000
           4       0.71      0.52      0.60      1000
           5       0.55      0.64      0.59      1000
           6       0.74      0.80      0.77      1000
           7       0.75      0.71      0.73      1000
           8       0.83      0.79      0.81      1000
           9       0.86      0.72      0.78      1000

    accuracy                           0.69     10000
   macro avg       0.70      0.69      0.69     10000
weighted avg       0.70      0.69      0.69     10000

Confusion Matrix:
[[782  20  75  16  11   6   5  10  55  20]
 [ 29 861   7  13   4   3   3   3  25  52]
 [ 67   9 633  53  57  79  68  21   9   4]
 [ 14  1